In [1]:
!pip install google-cloud-bigquery

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Stock_pred\stockanalysis-444512-3fd7af0c6e0a.json"

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project="stockanalysis-444512")

# Define dataset and table names
dataset_id = "stock_analysis"
table_id_ps = "stream_pubsubtobq"

# Full table reference
table_ref_ps = f"{client.project}.{dataset_id}.{table_id_ps}"
print(table_ref_ps)

# Check if dataset exists
try:
    dataset = client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} exists.")
except Exception:
    # Create the dataset if it doesn't exist
    dataset = bigquery.Dataset(f"{client.project}.{dataset_id}")
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {dataset_id} created.")

stockanalysis-444512.stock_analysis.stream_pubsubtobq
Dataset stock_analysis exists.


In [4]:
# Define dataset and table names
dataset_id = "stock_analysis"
table_id = "stream_mltobq"

# Full table reference
table_ref = f"{client.project}.{dataset_id}.{table_id}"
print(table_ref)

# Check if dataset exists
try:
    dataset = client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} exists.")
except Exception:
    # Create the dataset if it doesn't exist
    dataset = bigquery.Dataset(f"{client.project}.{dataset_id}")
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {dataset_id} created.")

stockanalysis-444512.stock_analysis.stream_mltobq
Dataset stock_analysis exists.


In [5]:
from google.cloud.exceptions import NotFound

try:
    # Check if the table exists
    table = client.get_table(table_ref)
    print(f"Table {table_id} exists.")

    # Check if the table has a schema
    if not table.schema:
        print(f"Table {table_id} has no schema. Adding schema...")
        schema = [
            bigquery.SchemaField("symbol", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("price", "FLOAT", mode="REQUIRED"),
            bigquery.SchemaField("timestamp", "TIMESTAMP", mode="REQUIRED"),
            bigquery.SchemaField("predicted_trend", "STRING", mode="NULLABLE"),
         ]
        table.schema = schema
        table = client.update_table(table, ["schema"])
        print(f"Schema added to table {table_id}.")
    else:
        print(f"Table {table_id} already has a schema.")
except NotFound:
    print(f"Table {table_id} does not exist. Creating it with schema...")
    schema = [
        bigquery.SchemaField("symbol", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("price", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("timestamp", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("predicted_trend", "STRING", mode="NULLABLE"),
    ]
    table = bigquery.Table(table_ref, schema=schema)
    table = client.create_table(table)
    print(f"Table {table_id} created with schema.")

Table stream_mltobq exists.
Table stream_mltobq already has a schema.


In [6]:
import tensorflow as tf

model = tf.keras.models.load_model('C:\\Stock_pred\\saved_model\\lstm_model.keras')

In [7]:
import yfinance as yf
import pandas as pd
import numpy as np

# Function to calculate features
def calculate_features(stock_data):
    stock_data['Return'] = stock_data['Close'].pct_change()  # Daily returns
    stock_data['SMA_5'] = stock_data['Close'].rolling(window=5).mean()  # 5-day Simple Moving Average
    stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()  # 20-day Simple Moving Average
    
    # Calculate RSI
    delta = stock_data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    stock_data['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    stock_data['Bollinger_Upper'] = stock_data['SMA_20'] + (stock_data['Close'].rolling(window=20).std() * 2)
    stock_data['Bollinger_Lower'] = stock_data['SMA_20'] - (stock_data['Close'].rolling(window=20).std() * 2)

    # Volume Change (percentage change)
    stock_data['Volume_Change'] = stock_data['Volume'].pct_change()

    # Drop NaN values
    return stock_data.dropna()

# Function to prepare features for the ML model
def prepare_features(stock_data):
    # Define the feature columns based on your model
    feature_columns = ['Return', 'SMA_5', 'SMA_20', 'RSI', 'Bollinger_Upper', 'Bollinger_Lower', 'Volume_Change']
    
    time_steps = 5
    if len(stock_data) < time_steps:
        raise ValueError("Not enough data to make a prediction. Requires at least 5 time steps.")

    # Get the last 5 entries for the specified features
    features = stock_data[feature_columns].tail(time_steps).values  # Last 5 time steps

    features = features.reshape((1, time_steps, len(feature_columns)))
    return features

# Function to predict the trend using the ML model
def predict_trend(features):
    prediction = model.predict(features)
    trend = "up" if prediction[0] == 1 else "down"  # Assuming 1 means "up", 0 means "down"
    return trend

In [8]:
# Fetch trending stock data
trending_symbols = ["TSLA", "AAPL", "MSFT", "GOOGL", "AMZN", "ORCL", "INTC", "NVDA", "META", "BABA"]  # Example top stocks
rows_to_insert = []

for symbol in trending_symbols:
    stock = yf.Ticker(symbol)
    stock_data = stock.history(period="3mo", interval="1d")
    
    if not stock_data.empty:
        # Calculate the features
        stock_data = calculate_features(stock_data)
        
        # Check if we have enough data to predict
        if len(stock_data) < 5:
            print(f"Not enough data for {symbol} after feature calculation.")
            continue
        
        latest_row = stock_data.iloc[-1]  # Get the latest stock data
        try:
            features = prepare_features(stock_data)
            predicted_trend = predict_trend(features)
            rows_to_insert.append({
                "symbol": symbol,
                "price": round(latest_row["Close"], 2),
                "timestamp": latest_row.name.isoformat(),
                "predicted_trend": predicted_trend
            })
        except ValueError as e:
            print(f"Error for {symbol}: {e}")

# Insert data into BigQuery
if rows_to_insert:
    errors = client.insert_rows_json(table, rows_to_insert)
    if errors:
        print(f"Errors occurred: {errors}")
    else:
        print("Trending stock data inserted successfully.")
else:
    print("No stock data available for insertion.")

# Display inserted data
print("Inserted Data:")
for row in rows_to_insert:
    print(row)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 887ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Trending stock data inserted successfully.
Inserted Data:
{'symbol': 'TSLA', 'price': 426.5, 'timestamp': '2025-01-17T00:00:00-05:00', 'predicted_trend': 'up'}
{'symbol': 'AAPL', 'price': 229.98, 'timestamp': '2025-01-17T00:00:00-05:00', 'predicted_trend': 'down'}
{'symbol': 'MSFT', 'price': 429.03, 'timestamp': '2025-01-17T00:00:00-05:00', 'predicted_trend': 'up'}
{'symbol': 'GOOGL', 'price': 196.0, 'timestamp': '2025-01-17T00:00:00-05:00', 'predicted_trend': 'down'}
{'symbol': 'AMZN', 'price': 225.94, 'timestamp': '2025-01-17T00:00:00-05:00', 'predicted_trend': 'down'}
{'symbol': 'ORCL', 'price': 161.03, '

In [9]:
# same function as that of the above cell i.e takes and stores predicted data but the data is taken from pubusb rather than from yfinance

"""
# Function to predict the trend using the ML model
def predict_trend(features):
    prediction = model.predict(features)
    trend = "up" if prediction[0] == 1 else "down"  # Assuming 1 means "up", 0 means "down"
    return trend

# Fetch data for predictions
query = """
#    SELECT symbol, price, timestamp
#    FROM `stockanalysis-444512.stock_analysis.stream_pubsubtobq`
#    WHERE NOT EXISTS (
#        SELECT 1
#        FROM `stockanalysis-444512.stock_analysis.stream_mltobq`
#        WHERE stream_pubsubtobq.symbol = stream_mltobq.symbol
#          AND stream_pubsubtobq.timestamp = stream_mltobq.timestamp    )
"""
rows = client.query(query).result()

rows_to_insert = []
for row in rows:
    try:
        stock_data = yf.download(row["symbol"], interval="5m", period="5d")
        features = prepare_features(stock_data)
        predicted_trend = predict_trend(features)
        rows_to_insert.append({
            "symbol": row["symbol"],
            "price": row["price"],
            "timestamp": row["timestamp"],
            "predicted_trend": predicted_trend,
        })
    except Exception as e:
        print(f"Error processing {row['symbol']}: {e}")

# Insert predictions into BigQuery
if rows_to_insert:
    errors = client.insert_rows_json(table_ref, rows_to_insert)
    if errors:
        print(f"BigQuery insert errors: {errors}")
    else:
        print("Predictions inserted into BigQuery.")
"""

'\nrows = client.query(query).result()\n\nrows_to_insert = []\nfor row in rows:\n    try:\n        stock_data = yf.download(row["symbol"], interval="5m", period="5d")\n        features = prepare_features(stock_data)\n        predicted_trend = predict_trend(features)\n        rows_to_insert.append({\n            "symbol": row["symbol"],\n            "price": row["price"],\n            "timestamp": row["timestamp"],\n            "predicted_trend": predicted_trend,\n        })\n    except Exception as e:\n        print(f"Error processing {row[\'symbol\']}: {e}")\n\n# Insert predictions into BigQuery\nif rows_to_insert:\n    errors = client.insert_rows_json(table_ref, rows_to_insert)\n    if errors:\n        print(f"BigQuery insert errors: {errors}")\n    else:\n        print("Predictions inserted into BigQuery.")\n'

In [10]:
# MERGE operation to update the table with any new data
if rows_to_insert:
    query = f"""
    MERGE {table_ref} T
    USING (
        SELECT * FROM UNNEST([
            {', '.join([f'Struct("{row["symbol"]}" AS symbol, {row["price"]} AS price, TIMESTAMP("{row["timestamp"]}") AS timestamp, "{row["predicted_trend"]}" AS predicted_trend)' for row in rows_to_insert])}
        ])
    ) S
    ON T.symbol = S.symbol AND T.price = S.price AND T.timestamp = S.timestamp
    WHEN NOT MATCHED THEN
    INSERT (symbol, price, timestamp, predicted_trend) VALUES (S.symbol, S.price, S.timestamp, S.predicted_trend)
    """
    
    # Execute the query
    query_job = client.query(query)
    query_job.result()  # Wait for the query to complete
    print("Data merged successfully.")

Data merged successfully.


In [11]:
# Query to display average price and total predictions for the last 7 days
query = f"""
SELECT symbol, AVG(price) as avg_price, COUNT(*) as total_predictions,
       predicted_trend
FROM {table_ref}
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY symbol, predicted_trend
ORDER BY symbol;
"""
query_job = client.query(query)
for row in query_job.result():
    print(row)

Row(('AAPL', 229.98, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('AMZN', 225.94, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('BABA', 85.12, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('GOOGL', 196.0, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('INTC', 21.49, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('META', 612.77, 1, 'up'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('MSFT', 429.03, 1, 'up'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('NVDA', 137.71, 1, 'down'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('ORCL', 161.03, 1, 'up'), {'symbol': 0, 'avg_price': 1, 'total_predictions': 2, 'predicted_trend': 3})
Row(('TSLA', 426.5